## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import scipy.sparse as sps
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import gc

from Challenge.paths import load_xgboost_cv_folds, XGBOOST_MODELS, XGBOOST_DATAFRAMES
from Challenge.utils import load_models

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [4]:
URM_inner, URM_outer, folds = load_xgboost_cv_folds()

## **Recommeder List**

In [5]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask

models_mapping = {
    'TopPop': TopPop,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_jaccard': ItemKNNCFRecommender,
    'ItemKNN_asymmetric': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'ItemKNN_dice': ItemKNNCFRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_jaccard': UserKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'UserKNN_dice': UserKNNCFRecommender,
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'EASE_R': EASE_R_Recommender,
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    # 'IALS': IALSRecommender,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    'MatrixFactorization_SVDpp': MatrixFactorization_SVDpp_Cython,
    
    'SLIM_BPR': SLIM_BPR_Cython,
    'NMF': NMFRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
}

candidate_mapping = {
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'RP3beta': RP3betaRecommender,
    # 'IALS': IALSRecommender,
    'TopPop': TopPop
}

candidate_cutoff = {
    'SLIMElasticNet': 80,
    'ItemKNN_tversky': 50,
    'UserKNN_asymmetric': 50,
    'RP3beta': 50,
    # 'IALS': 50,
    'TopPop': 50
}

## **Functions**

In [6]:
def get_user_batches(user_ids, batch_size=1000):
    for i in range(0, len(user_ids), batch_size):
        yield user_ids[i:i + batch_size]

In [7]:
def generate_candidates(URM, model_folder):
    model_folder = os.path.join(XGBOOST_MODELS, model_folder)
    candidate_models = load_models(URM, candidate_mapping, model_folder=model_folder)

    n_users, n_items = URM.shape
    user_ids = np.arange(n_users)
    
    dfs_to_concat = []
    for model_name, recommender in candidate_models:
        cutoff = candidate_cutoff[model_name]
        print(f"Generating candidates for model: {model_name}")
        
        recommendations_model = recommender.recommend(user_ids, cutoff=cutoff)
        
        df_model = pd.DataFrame({
            "UserID": user_ids,
            "ItemID": recommendations_model
        })

        df_model = df_model.explode("ItemID")

        dfs_to_concat.append(df_model)

    
    # Concatenate
    print("Concatenating and removing duplicates...")
    df = pd.concat(dfs_to_concat, ignore_index=True)
    
    # Ensure correct data types (Explode can sometimes create objects)
    df["ItemID"] = df["ItemID"].astype(int)
    
    # Drop duplicates
    df = df.drop_duplicates(subset=["UserID", "ItemID"])

    return df

In [8]:
def add_models_features(df, URM, model_folder, batch_size=500):
    model_folder = os.path.join(XGBOOST_MODELS, model_folder)
    models = load_models(URM, models_mapping, model_folder=model_folder)

    # Sort and clean index immediately.
    # This creates the only necessary copy of the data structure.
    # After this, 'df' has index 0..N
    df = df.sort_values("UserID").reset_index(drop=True)
    
    cand_users = df['UserID'].values
    cand_items = df['ItemID'].values
    
    N_CANDIDATES = len(df)
    unique_users = df['UserID'].unique()

    for label, recommender in models:
        print(f"Processing features for model: {label}")
        
        # Pre-allocate arrays
        scores_final = np.zeros(N_CANDIDATES, dtype=np.float32)
        ranks_final = np.zeros(N_CANDIDATES, dtype=np.int32)
        
        for user_batch in tqdm(list(get_user_batches(unique_users, batch_size)), desc=f"Batches {label}", leave=False):
            
            # Compute Scores
            scores_batch = recommender._compute_item_score(user_id_array=user_batch)

            # Normalize
            norm_factor = np.linalg.norm(scores_batch, np.inf, axis=1, keepdims=True)
            norm_factor[norm_factor == 0] = 1.0 
            linf_scores_batch = scores_batch / norm_factor

            # Remove seen
            for i, user_id in enumerate(user_batch):
                linf_scores_batch[i, :] = recommender._remove_seen_on_scores(user_id, linf_scores_batch[i, :])

            # Calculate Ranks
            # argsort sorts ascending, so we use [::-1] to get descending (highest score first)
            # This returns INDICES of items. 
            # shape: (batch_size, n_items)
            rank_order = np.argsort(linf_scores_batch, axis=1)[:, ::-1]
            
            # We need the inverse mapping: item_id -> rank_position
            n_batch, n_items = scores_batch.shape
            rank_matrix_batch = np.empty((n_batch, n_items), dtype=np.int32)
            
            # Fancy numpy trick to invert the permutation vectors in one go
            # Arrays of shape (n_batch, 1) needed for broadcasting
            row_indices = np.arange(n_batch)[:, None] 
            rank_matrix_batch[row_indices, rank_order] = np.arange(n_items)
            
            # Fast Lookup using SearchSorted (requires sorted cand_users)
            start_pos = np.searchsorted(cand_users, user_batch[0], side='left')
            end_pos = np.searchsorted(cand_users, user_batch[-1], side='right')
            
            # Slice the global candidate arrays
            batch_cand_items = cand_items[start_pos:end_pos]
            batch_cand_users = cand_users[start_pos:end_pos]
            
            # We need to map global UserIDs to local batch indices (0 to BATCH_SIZE-1)
            local_user_indices = np.searchsorted(user_batch, batch_cand_users)
            
            # Extract Values
            scores_final[start_pos:end_pos] = linf_scores_batch[local_user_indices, batch_cand_items]
            ranks_final[start_pos:end_pos] = rank_matrix_batch[local_user_indices, batch_cand_items]
            
            # Explicit cleanup
            del scores_batch, linf_scores_batch, rank_matrix_batch, rank_order

        # Assign directly to DF
        df[f"{label}_Score"] = scores_final
        df[f"{label}_RankPosition"] = ranks_final
        df[f"{label}_Recommended"] = (ranks_final < 20).astype(int)
        
        # Free memory immediately
        del scores_final, ranks_final
        gc.collect()

    return df

In [9]:
def calculate_item_item_features_fast(df, URM, model_folder):
    # Check df is sorted by UserID
    assert df['UserID'].is_monotonic_increasing, "DataFrame must be sorted by UserID in increasing order."

    # Load only distinct model types
    similarity_models = load_models(
        URM,
        {
            'ItemKNN_tversky': ItemKNNCFRecommender, 
            'RP3beta': RP3betaRecommender,
            'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender 
        },
        model_folder=os.path.join(XGBOOST_MODELS, model_folder)
    )

    # Pre-calculate mapping: UserID -> [List of Candidate ItemIDs]
    # We use numpy split for max speed
    user_ids = df['UserID'].values
    item_ids = df['ItemID'].values
    
    # Find indices where user changes
    unique_users, user_starts = np.unique(user_ids, return_index=True)
    # Map UserID to (start_index, end_index) in the sorted arrays
    user_map = {}
    for i, user_id in enumerate(unique_users):
        start = user_starts[i]
        end = user_starts[i+1] if i + 1 < len(unique_users) else len(user_ids)
        user_map[user_id] = (start, end)
    
    # Prepare result dictionary
    N_ROWS = len(df)
    
    for label, recommender in similarity_models:
        print(f"Extracting MAX similarity for {label}...")
        
        # Get Sparse Matrix
        W_sparse = recommender.W_sparse
        if not sps.issparse(W_sparse):
            W_sparse = sps.csr_matrix(W_sparse)
            
        # Feature arrays
        max_sims = np.zeros(N_ROWS, dtype=np.float32)
        std_sims = np.zeros(N_ROWS, dtype=np.float32)
        
        # Iterate over unique users in the candidates
        for user_id in tqdm(unique_users):
            start, end = user_map[user_id]
            
            # Get Seen Items for this user (Indices)
            seen_items = URM.indices[URM.indptr[user_id]:URM.indptr[user_id+1]]
            
            if len(seen_items) == 0:
                continue
                
            # Get Candidate Items for this user
            cand_items = item_ids[start:end]
            
            # --- THE CORE OPTIMIZATION ---
            # Instead of .toarray(), we slice sparse matrix
            # Submatrix: Rows=Candidates, Cols=SeenItems
            # This is efficient because W is CSR (fast row slicing)
            sub_W = W_sparse[cand_items, :][:, seen_items]
            
            # Check if sub_W is effectively empty
            if sub_W.nnz > 0:
                # Max similarity to any seen item
                max_sims[start:end] = sub_W.max(axis=1).toarray().flatten()
                
                # For Std, you usually need to densify (slower)
                dense_batch = sub_W.toarray()
                std_sims[start:end] = dense_batch.std(axis=1)

        # Assign directly to DF
        df[f'{label}_MaxSim'] = max_sims
        df[f'{label}_StdSim'] = std_sims
    
    return df

In [10]:
def add_embedding_features_batched(training_dataframe, URM_train, model_folder, 
                                   pca_components=5, n_item_clusters=10, n_user_clusters=10, 
                                   batch_size=200000):
    
    # 1. Handle Index (Safety First)
    if training_dataframe.index.name == 'UserID':
        print("Resetting index to restore UserID column...")
        training_dataframe = training_dataframe.reset_index()

    # 2. Load Model & Factors
    print("Loading IALS model...")
    model_path = os.path.join(XGBOOST_MODELS, model_folder)
    recommender = IALSRecommender(URM_train)
    recommender.load_model(model_path, "IALS.zip")
    
    user_factors = recommender.USER_factors.astype(np.float32)
    item_factors = recommender.ITEM_factors.astype(np.float32)
    
    # --- PART A: CLUSTERING (Low Memory, do it all at once) ---
    print("Calculating K-Means Clusters...")
    
    # Cluster Users
    kmeans_users = KMeans(n_clusters=n_user_clusters, random_state=42, n_init=10).fit(user_factors)
    # Map directly using pandas map (fast)
    training_dataframe['User_Cluster'] = kmeans_users.labels_[training_dataframe['UserID'].values.astype(int)]
    
    # Cluster Items
    kmeans_items = KMeans(n_clusters=n_item_clusters, random_state=42, n_init=10).fit(item_factors)
    training_dataframe['Item_Cluster'] = kmeans_items.labels_[training_dataframe['ItemID'].values.astype(int)]
    
    del kmeans_users, kmeans_items
    gc.collect()

    # --- PART B: PCA SETUP (Fit on a small random sample) ---
    print("Fitting PCA on random sample...")
    
    # Pick 50k random indices to learn the PCA transformation
    # We do this OUTSIDE the loop so the "Coordinate System" is consistent
    n_samples = min(50000, len(training_dataframe))
    sample_indices = np.random.choice(len(training_dataframe), n_samples, replace=False)
    
    sample_users = training_dataframe['UserID'].values[sample_indices].astype(int)
    sample_items = training_dataframe['ItemID'].values[sample_indices].astype(int)
    
    sample_interactions = user_factors[sample_users] * item_factors[sample_items]
    
    pca = PCA(n_components=pca_components)
    pca.fit(sample_interactions)
    
    del sample_interactions, sample_users, sample_items
    gc.collect()

    # --- PART C: BATCHED CALCULATION (Euclidean & PCA Transform) ---
    print(f"Processing Batches (Batch Size: {batch_size})...")
    
    num_rows = len(training_dataframe)
    
    # 1. Pre-allocate columns with float32 (saves 50% RAM compared to float64)
    # We initialize with zeros
    training_dataframe['IALS_EuclideanDist'] = np.zeros(num_rows, dtype=np.float32)
    for i in range(pca_components):
        training_dataframe[f'IALS_PCA_{i}'] = np.zeros(num_rows, dtype=np.float32)
    
    # 2. Get the numpy array views of the dataframe columns for fast writing
    # (Writing to these arrays updates the dataframe directly)
    user_ids_all = training_dataframe['UserID'].values.astype(int)
    item_ids_all = training_dataframe['ItemID'].values.astype(int)
    
    dist_column = training_dataframe['IALS_EuclideanDist'].values
    pca_columns = [training_dataframe[f'IALS_PCA_{i}'].values for i in range(pca_components)]
    
    # 3. The Batch Loop
    for start_idx in tqdm(range(0, num_rows, batch_size)):
        end_idx = min(start_idx + batch_size, num_rows)
        
        # A. Get Vectors for this batch only
        batch_u_idx = user_ids_all[start_idx:end_idx]
        batch_i_idx = item_ids_all[start_idx:end_idx]
        
        batch_u_vecs = user_factors[batch_u_idx]
        batch_i_vecs = item_factors[batch_i_idx]
        
        # B. Euclidean Distance
        # Vectorized batch calculation
        diff = batch_u_vecs - batch_i_vecs
        batch_dist = np.linalg.norm(diff, axis=1)
        
        # Assign to main array
        dist_column[start_idx:end_idx] = batch_dist
        
        # C. PCA Transform
        # Calculate interaction for batch
        batch_interaction = batch_u_vecs * batch_i_vecs
        
        # Transform using the pre-fitted PCA
        batch_pca = pca.transform(batch_interaction)
        
        # Assign columns
        for k in range(pca_components):
            pca_columns[k][start_idx:end_idx] = batch_pca[:, k]
            
        # D. Cleanup
        del batch_u_vecs, batch_i_vecs, diff, batch_interaction, batch_pca, batch_dist
        # Optional: gc.collect() every few batches if RAM is extremely tight
        # if start_idx % (batch_size * 5) == 0: gc.collect()

    print("Embedding features added successfully.")
    return training_dataframe

In [11]:
def add_aggregate_features_stats(df):
    # Consensus Features
    recommended_columns = [col for col in df.columns if col.endswith('_Recommended')]
    df['Counter_Recommended'] = df[recommended_columns].sum(axis=1).astype(int)

    # Rank Position Statistics
    position_columns = [col for col in df.columns if col.endswith('_RankPosition')]
    
    df['Mean_RankPosition'] = df[position_columns].mean(axis=1)
    df['Std_RankPosition'] = df[position_columns].std(axis=1)
    df['Skew_RankPosition'] = df[position_columns].skew(axis=1)
    df['Kurtosis_RankPosition'] = df[position_columns].kurtosis(axis=1)
    
    # Score Statistics
    score_columns = [col for col in df.columns if col.endswith('_Score')]

    df['Mean_Score'] = df[score_columns].mean(axis=1)
    df['Std_Score'] = df[score_columns].std(axis=1)
    df['Skew_Score'] = df[score_columns].skew(axis=1)
    df['Kurtosis_Score'] = df[score_columns].kurtosis(axis=1)
    
    return df

In [12]:
def add_user_stats(df, URM):
    # We need to map UserID and ItemID to the URM indices
    user_ids = df['UserID'].values
    item_ids = df['ItemID'].values
    
    # User Profile Length
    user_profile_len = np.ediff1d(URM.indptr)
    df['User_Profile_Len'] = user_profile_len[user_ids]
    
    # Item Global Popularity
    item_popularity = np.ediff1d(URM.tocsc().indptr)
    df['Item_Global_Popularity'] = item_popularity[item_ids]
    
    return df

In [13]:
def sanity_check(df, verbose=True):
    print("--- STARTING SANITY CHECK ---")
    problems_found = False
    
    # 1. Check for Missing Values (NaN)
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print("\n[CRITICAL] NaN Values Found:")
        print(null_counts[null_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No NaNs found.")

    # 2. Check for Infinite Values (inf / -inf)
    # Common issue when normalizing by zero variance or dividing scores
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum()
    if inf_counts.sum() > 0:
        print("\n[CRITICAL] Infinite Values Found (Division by Zero?):")
        print(inf_counts[inf_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No Infinite values found.")

    # 3. Check for Duplicates (UserID, ItemID)
    # Stacking fails if you have multiple rows for the same User-Item pair
    if df.duplicated(subset=['UserID', 'ItemID']).any():
        n_dupes = df.duplicated(subset=['UserID', 'ItemID']).sum()
        print(f"\n[CRITICAL] Duplicate (UserID, ItemID) pairs found: {n_dupes}")
        problems_found = True
    else:
        if verbose: print("[OK] Keys (UserID, ItemID) are unique.")

    # 4. Check Data Types (IDs must be int)
    # Merges (pd.merge) often convert ints to float if there were missing keys initially
    if df['UserID'].dtype not in [int, np.int32, np.int64]:
        print(f"\n[WARNING] UserID is {df['UserID'].dtype}, expected int. (Did a merge fail?)")
        # Auto-fix attempt
        # df['UserID'] = df['UserID'].astype(int) 
    
    if df['ItemID'].dtype not in [int, np.int32, np.int64]:
        print(f"\n[WARNING] ItemID is {df['ItemID'].dtype}, expected int.")

    # 5. Check for Constant Columns (Zero Variance)
    # These crash some implementations of Normalization and add no info to XGBoost
    std_devs = df[numeric_cols].std()
    constant_cols = std_devs[std_devs == 0].index.tolist()
    if len(constant_cols) > 0:
        print("\n[WARNING] The following columns have ZERO variance (Constant values):")
        print(constant_cols)
        print("Recommendation: Drop them.")
    
    # 6. Check Logic (Ranks shouldn't be negative)
    rank_cols = [c for c in df.columns if 'Rank' in c and 'Skew' not in c and 'Kurtosis' not in c]
    if rank_cols:
        min_ranks = df[rank_cols].min()
        if (min_ranks < 0).any():
             print("\n[CRITICAL] Negative Ranks found (Logic Error):")
             print(min_ranks[min_ranks < 0])
             problems_found = True

    if problems_found:
        print("\n--- SANITY CHECK FAILED: Fix errors before training ---")
        # raise ValueError("Data Integrity Issues Found") # Uncomment to force stop
    else:
        print("\n--- SANITY CHECK PASSED: Data is clean ---")

    return not problems_found

In [14]:
def optimize_dataframe_types(df):
    # 1. Downcast Integers (UserID, ItemID, Ranks)
    # int64 -> int32 (saves 50% RAM)
    ints = df.select_dtypes(include=['int64', 'int32', 'int']).columns
    df[ints] = df[ints].apply(pd.to_numeric, downcast='integer')
    
    # 2. Downcast Floats (Scores, Similarities)
    # float64 -> float32 (saves 50% RAM, XGBoost doesn't need float64)
    floats = df.select_dtypes(include=['float64', 'float']).columns
    df[floats] = df[floats].apply(pd.to_numeric, downcast='float')
    
    return df

## **Training Dataframe**

In [15]:
df_list = []

for i, (URM_train, URM_val) in enumerate(folds):
    models_folder = os.path.join("train_features", "folds", f"fold_{i}")

    # Generate Candidates
    print(f"\n=== FOLD {i}: Generating Candidates ===")
    df = generate_candidates(URM_train, models_folder)
    print(f"Candidates Generated: {len(df)}")

    # Add Label Column
    print(f"\n=== FOLD {i}: Adding Labels ===")
    URM_validation_coo = sps.coo_matrix(URM_val)

    correct_recommendations = pd.DataFrame({"UserID": URM_validation_coo.row,
                                            "ItemID": URM_validation_coo.col})

    df = pd.merge(df, correct_recommendations, on=['UserID','ItemID'], how='left', indicator='Exist')
    df["Label"] = df["Exist"] == "both"
    df.drop(columns = ['Exist'], inplace=True)

    print(f"Positive Labels: {df['Label'].sum()} / {len(df)} ({100.0 * df['Label'].mean():.4f}%)")

    # Add Model Features
    print(f"\n=== FOLD {i}: Adding Model Features ===")
    df = add_models_features(df, URM_train, models_folder, batch_size=500)
    print(f"Features after model addition: {len(df.columns)}")

    # Add Item-Item Similarity Features
    print(f"\n=== FOLD {i}: Adding Item-Item Similarity Features ===")
    df = calculate_item_item_features_fast(df, URM_train, models_folder)
    print(f"Features after item-item similarity addition: {len(df.columns)}")

    # Add Embedding Features
    # print(f"\n=== FOLD {i}: Adding Embedding Features ===")
    # df = add_embedding_features_batched(df, URM_train, models_folder, 
    #                                    pca_components=5, n_item_clusters=10, n_user_clusters=10, 
    #                                    batch_size=200000)
    # print(f"Features after embedding addition: {len(df.columns)}")
    
    # Add Aggregate Features
    print(f"\n=== FOLD {i}: Adding Aggregate Features ===")
    df = add_aggregate_features_stats(df)
    print(f"Features after aggregate addition: {len(df.columns)}")

    # Add User Stats
    print(f"\n=== FOLD {i}: Adding User and Item Stats ===")
    df = add_user_stats(df, URM_train)
    print(f"Features after user/item stats addition: {len(df.columns)}")

    # Sanity Check
    print(f"\n=== FOLD {i}: Running Sanity Check ===")
    success = sanity_check(df)
    if not success:
        print(f"Sanity check failed for fold {i}.")
        break

    # Optimize Data Types
    print(f"\n=== FOLD {i}: Optimizing Data Types ===")
    df = optimize_dataframe_types(df)
    df_list.append(df)


=== FOLD 0: Generating Candidates ===
Model found: SLIMElasticNet
Model found: ItemKNN_tversky
Model found: UserKNN_asymmetric
Model found: RP3beta
Model found: TopPop
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0SLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Generating candidates for model: SLIMElasticNet
Unloading SLIMElasticNet...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0ItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Generating candidates for model: ItemKNN_tversky
Unloading ItemKNN_tversky...
Loading UserKNN_asymmetric...
UserKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0UserKNN_asymmetric'
UserKNNCFRecommender: Loading complete
Generating candidates for model: UserKNN_asymmetric
Unloading UserKNN_as

Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0ItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_cosine


Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0ItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_jaccard


Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0ItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_asymmetric


Unloading ItemKNN_asymmetric...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_0ItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_tversky


...

Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4RP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:04<00:00, 5881.41it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/folds/fold_4SLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:07<00:00, 3702.26it/s]


Unloading SLIMElasticNet...
Features after item-item similarity addition: 72

=== FOLD 4: Adding Aggregate Features ===
Features after aggregate addition: 81

=== FOLD 4: Adding User and Item Stats ===
Features after user/item stats addition: 83

=== FOLD 4: Running Sanity Check ===
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

--- SANITY CHECK PASSED: Data is clean ---

=== FOLD 4: Optimizing Data Types ===


In [16]:
# Merge all folds
full_df = pd.concat(df_list, ignore_index=True)
del df_list
gc.collect()

0

In [18]:
save_path = os.path.join(XGBOOST_DATAFRAMES, "training_data.parquet")

full_df.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

Saved successfully to: /home/luigi/RecSys/xg_boost_data/dataframes/training_data.parquet


In [ ]:
del full_df
gc.collect()

922

## **Validation Dataframe**

In [15]:
models_folder = os.path.join("train_features", "inner")

URM_train = URM_inner
URM_val = URM_outer

In [16]:
# Generate Candidates
print(f"\n=== Generating Candidates ===")
df = generate_candidates(URM_train, models_folder)
print(f"Candidates Generated: {len(df)}")

# Add Label Column
print(f"\n=== Adding Labels ===")
URM_validation_coo = sps.coo_matrix(URM_val)

correct_recommendations = pd.DataFrame({"UserID": URM_validation_coo.row,
                                        "ItemID": URM_validation_coo.col})

df = pd.merge(df, correct_recommendations, on=['UserID','ItemID'], how='left', indicator='Exist')
df["Label"] = df["Exist"] == "both"
df.drop(columns = ['Exist'], inplace=True)

print(f"Positive Labels: {df['Label'].sum()} / {len(df)} ({100.0 * df['Label'].mean():.4f}%)")

# Add Model Features
print(f"\n=== Adding Model Features ===")
df = add_models_features(df, URM_train, models_folder, batch_size=500)
print(f"Features after model addition: {len(df.columns)}")

# Add Item-Item Similarity Features
print(f"\n=== Adding Item-Item Similarity Features ===")
df = calculate_item_item_features_fast(df, URM_train, models_folder)
print(f"Features after item-item similarity addition: {len(df.columns)}")

# Add Embedding Features
# print(f"\n=== Adding Embedding Features ===")
# df = add_embedding_features_batched(df, URM_train, models_folder, 
#                                    pca_components=5, n_item_clusters=10, n_user_clusters=10, 
#                                    batch_size=200000)
# print(f"Features after embedding addition: {len(df.columns)}")

# Add Aggregate Features
print(f"\n=== Adding Aggregate Features ===")
df = add_aggregate_features_stats(df)
print(f"Features after aggregate addition: {len(df.columns)}")

# Add User Stats
print(f"\n=== Adding User and Item Stats ===")
df = add_user_stats(df, URM_train)
print(f"Features after user/item stats addition: {len(df.columns)}")

# Sanity Check
print(f"\n=== Running Sanity Check ===")
assert sanity_check(df), "Sanity check failed for inner fold."

# Optimize Data Types
print(f"\n=== Optimizing Data Types ===")
df = optimize_dataframe_types(df)


=== Generating Candidates ===
Model found: SLIMElasticNet
Model found: ItemKNN_tversky
Model found: UserKNN_asymmetric
Model found: RP3beta
Model found: TopPop
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Generating candidates for model: SLIMElasticNet
Unloading SLIMElasticNet...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Generating candidates for model: ItemKNN_tversky
Unloading ItemKNN_tversky...
Loading UserKNN_asymmetric...
UserKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerUserKNN_asymmetric'
UserKNNCFRecommender: Loading complete
Generating candidates for model: UserKNN_asymmetric
Unloading UserKNN_asymmetric...
Loading RP3beta..

Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_cosine


Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_jaccard


Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_asymmetric


Unloading ItemKNN_asymmetric...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_tversky


...

Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:04<00:00, 6270.94it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/innerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:08<00:00, 3361.50it/s]


Unloading SLIMElasticNet...
Features after item-item similarity addition: 72

=== Adding Aggregate Features ===
Features after aggregate addition: 81

=== Adding User and Item Stats ===
Features after user/item stats addition: 83

=== Running Sanity Check ===
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

--- SANITY CHECK PASSED: Data is clean ---

=== Optimizing Data Types ===


In [17]:
df

,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,6411,False,0.081851,717,0,0.701361,5,1,0.681310,...,407.142853,1399.072754,4.281490,18.897297,0.447156,0.278489,-1.061027,3.246427,80,874
1,0,2168,False,0.078760,750,0,0.540989,40,0,0.491260,...,358.047607,936.414124,3.650616,14.062557,0.333984,0.222666,0.733968,1.210770,80,841
2,0,6098,True,0.082225,713,0,0.708915,3,1,0.740307,...,124.571426,279.955811,2.169494,3.165847,0.503251,0.205144,0.271329,0.168902,80,878
3,0,3049,False,0.128114,412,0,0.693846,6,1,0.457670,...,321.095245,918.941772,4.353480,19.441338,0.310892,0.245103,0.985542,0.890398,80,1368
4,0,2399,False,0.442124,27,0,0.639774,15,1,0.261722,...,362.523804,1433.629395,4.515592,20.544024,0.405237,0.275703,-1.123605,4.698762,80,4721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3756433,27094,4180,False,0.367297,48,0,0.042708,2024,0,0.000000,...,3173.142822,2376.120117,0.292561,-1.572491,0.066819,0.232602,3.163013,11.877620,250,3922
3756434,27094,3236,False,0.593651,8,1,0.194622,524,0,0.114155,...,2111.714355,2253.487793,1.051186,-0.458617,0.134725,0.246047,2.432874,6.639362,250,6339
3756435,27094,4615,False,0.004402,4126,0,0.144729,767,0,0.037827,...,1697.571411,1721.538208,1.812891,2.658815,0.099026,0.205823,2.965396,11.336728,250,47
3756436,27094,6584,False,0.412343,31,0,0.112825,1005,0,0.000000,...,2309.666748,1955.347290,0.944244,-0.389569,0.105150,0.219539,3.446590,12.917460,250,4403


In [18]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "validation_data.parquet")

df.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

Saved successfully to: /home/luigi/RecSys/xg_boost_data/dataframes/validation_data.parquet


In [19]:
del df
gc.collect()

20

## **Prediction Dataframe**

In [20]:
models_folder = os.path.join("train_features", "outer")

URM_train = URM_inner + URM_outer

In [21]:
# Generate Candidates
print(f"\n=== Generating Candidates ===")
df = generate_candidates(URM_train, models_folder)
print(f"Candidates Generated: {len(df)}")

# Add Model Features
print(f"\n=== Adding Model Features ===")
df = add_models_features(df, URM_train, models_folder, batch_size=500)
print(f"Features after model addition: {len(df.columns)}")

# Add Item-Item Similarity Features
print(f"\n=== Adding Item-Item Similarity Features ===")
df = calculate_item_item_features_fast(df, URM_train, models_folder)
print(f"Features after item-item similarity addition: {len(df.columns)}")

# Add Embedding Features
# print(f"\n=== Adding Embedding Features ===")
# df = add_embedding_features_batched(df, URM_train, models_folder, 
#                                    pca_components=5, n_item_clusters=10, n_user_clusters=10, 
#                                    batch_size=200000)
# print(f"Features after embedding addition: {len(df.columns)}")

# Add Aggregate Features
print(f"\n=== Adding Aggregate Features ===")
df = add_aggregate_features_stats(df)
print(f"Features after aggregate addition: {len(df.columns)}")

# Add User Stats
print(f"\n=== Adding User and Item Stats ===")
df = add_user_stats(df, URM_train)
print(f"Features after user/item stats addition: {len(df.columns)}")

# Sanity Check
print(f"\n=== Running Sanity Check ===")
assert sanity_check(df), "Sanity check failed for inner fold."

# Optimize Data Types
print(f"\n=== Optimizing Data Types ===")
df = optimize_dataframe_types(df)


=== Generating Candidates ===
Model found: SLIMElasticNet
Model found: ItemKNN_tversky
Model found: UserKNN_asymmetric
Model found: RP3beta
Model found: TopPop
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Generating candidates for model: SLIMElasticNet
Unloading SLIMElasticNet...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Generating candidates for model: ItemKNN_tversky
Unloading ItemKNN_tversky...
Loading UserKNN_asymmetric...
UserKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerUserKNN_asymmetric'
UserKNNCFRecommender: Loading complete
Generating candidates for model: UserKNN_asymmetric
Unloading UserKNN_asymmetric...
Loading RP3beta..

Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_cosine


Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_jaccard


Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_asymmetric


Unloading ItemKNN_asymmetric...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_tversky


...

Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:04<00:00, 6074.05it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/train_features/outerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:08<00:00, 3277.26it/s]


Unloading SLIMElasticNet...
Features after item-item similarity addition: 71

=== Adding Aggregate Features ===
Features after aggregate addition: 80

=== Adding User and Item Stats ===
Features after user/item stats addition: 82

=== Running Sanity Check ===
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

--- SANITY CHECK PASSED: Data is clean ---

=== Optimizing Data Types ===


In [22]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "prediction_data.parquet")

df.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

Saved successfully to: /home/luigi/RecSys/xg_boost_data/dataframes/prediction_data.parquet


In [23]:
del df
gc.collect()

0